In [68]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

from sklearn.preprocessing import LabelEncoder
from imutils import paths
from PIL import Image
from numpy import asarray
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.svm import SVC
from sklearn.manifold import TSNE
from sklearn.model_selection import GridSearchCV


In [69]:
def get_features(model,X):
    hidden = X
    for i in range(len(model.coefs_)-1):
        hidden = np.dot(hidden,model.coefs_[i]) + model.intercepts_[i]
        hidden = np.maximum(hidden,0)
    return hidden

def lbp_numpy(gray):
    h,w = gray.shape
    lbp_image = np.zeros((h,w),dtype=np.uint8)
    neighbors = [
        (-1,-1),(-1,0),(-1,1),
        (0,1),
        (1,1),(1,0),(1,-1),
        (0,-1)
    ]
    for y in range(1,h-1):
        for x in range(1,w-1):
            center = gray[y,x]
            binary_string = ''
            for dy,dx in neighbors:
                neighbor = gray[y+dy,x+dx]
                binary_string += '1' if neighbor > center else '0'
            lbp_image [y,x] = int(binary_string,2)
    hist,_ = np.histogram(lbp_image.ravel(),bins=256,range=(0,256))
    hist = hist.astype("float") / (hist.sum()+ 1e-6)
    return hist

def hog_numpy(gray,cell_size=8,block_size=2,bins=9):
    h, w = gray.shape
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=1)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=1)
    magnitude = np.sqrt(gx**2 + gy**2)
    angle = (np.arctan2(gy, gx) * 180 / np.pi) % 180
    cell_x = w // cell_size
    cell_y = h // cell_size
    hist = np.zeros((cell_y, cell_x, bins))
    bin_width = 180 / bins
    for i in range(cell_y):
        for j in range(cell_x):
            mag_block = magnitude[i*cell_size:(i+1)*cell_size, j*cell_size:(j+1)*cell_size]
            ang_block = angle[i*cell_size:(i+1)*cell_size, j*cell_size:(j+1)*cell_size]
            for y in range(cell_size):
                for x in range(cell_size):
                    mag = mag_block[y,x]
                    ang = ang_block[y,x]
                    bin_idx = int(ang // bin_width)
                    hist[i,j,bin_idx] += mag
    hog_features = []
    for i in range(cell_y - block_size + 1):
        for j in range(cell_x - block_size + 1):
            block = hist[i:i+block_size, j:j+block_size].ravel()
            norm = np.linalg.norm(block) + 1e-6
            hog_features.append(block / norm)
    hog_features = np.concatenate(hog_features)
    return hog_features
    
def Open_and_resize(image_path):
    image = Image.open(image_path).convert("RGB")
    resize_image = image.resize((128,128))
    data = asarray(resize_image)
    return data

def good_image(arr_image,cls=None):
    if cls=="Horse":
        return np.var(arr_image)>1000        
    return np.var(arr_image)>3000        
        
dataset = r'C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\PetImages'
image_list = os.listdir(dataset)
print(image_list)
X=[]
Y=[]
for i in image_list:
    folders_type = os.path.join(dataset, i)
    folder_name = os.path.basename(folders_type)
    print(folders_type)
    if "Cat" in str(folder_name):
        print("cat")
        for cat_img in tqdm(os.listdir(folders_type)):
             cat_image_path = os.path.join(folders_type, cat_img)
             try:
                     cat_arr=Open_and_resize(cat_image_path)
                     if cat_arr.shape ==(128,128,3) and good_image(cat_arr) :
                         gray = np.dot(cat_arr[...,:3], [0.299, 0.587, 0.114]).astype(np.uint8)
                         lbp_feat = lbp_numpy(gray)
                         hog_feat = hog_numpy(gray)
                         features = np.concatenate([lbp_feat, hog_feat])
                         X.append(features)
                         Y.append("Cat") 
             except (UnidentifiedImageError, OSError):
                    print("Can't open:", cat_image_path)
    elif "Dog" in str(folder_name):
         print("dog")
         for dog_img in tqdm(os.listdir(folders_type)):
             dog_image_path = os.path.join(folders_type,dog_img)
             try:
                     dog_arr=Open_and_resize(dog_image_path)
                     if dog_arr.shape ==(128,128,3) and good_image(dog_arr):
                         gray = np.dot(dog_arr[...,:3], [0.299, 0.587, 0.114]).astype(np.uint8)
                         lbp_feat = lbp_numpy(gray)
                         hog_feat = hog_numpy(gray)
                         features = np.concatenate([lbp_feat, hog_feat])
                         X.append(features)
                         Y.append("Dog")
             except (UnidentifiedImageError, OSError):
                     print("Can't open:", dog_image_path)
X = np.array(X)

['Cat', 'Dog', 'Horse']
C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\PetImages\Cat
cat


100%|██████████████████████████████████████████████████████████████████████████████| 3194/3194 [01:18<00:00, 40.68it/s]


C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\PetImages\Dog
dog


100%|██████████████████████████████████████████████████████████████████████████████| 3201/3201 [01:12<00:00, 44.03it/s]


C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\PetImages\Horse


In [70]:
lable = LabelEncoder()
y_lable = lable.fit_transform(Y)

In [71]:
x_train,x_test,y_train,y_test=train_test_split(X,y_lable,test_size=0.2,random_state=42,stratify=y_lable)

In [ ]:
svm = SVC(kernel="rbf", C=1, gamma="scale", class_weight="balanced")
svm.fit(x_train, y_train)

In [ ]:
y_predict = svm.predict(x_test)

print("Confusion matrix results:\n",confusion_matrix(y_predict,y_test))
print("\nClassification report of model:\n",classification_report(y_predict,y_test))
print("Accuracy score:",100*accuracy_score(y_predict,y_test))

In [ ]:
def predict_image(image_path, svm_model, scaler=None):
    try:
        arr_image = Open_and_resize(image_path)
        if arr_image.shape != (128,128,3):
            print("Invalid image shape:", arr_image.shape)
            return None
            
        gray = np.dot(arr_image[...,:3], [0.299, 0.587, 0.114]).astype(np.uint8)
        lbp_feat = lbp_numpy(gray)
        hog_feat = hog_numpy(gray)

        features = np.concatenate([lbp_feat, hog_feat]).reshape(1, -1)

        decision = svm_model.decision_function(features)
        threshold = 0.5
        if np.max(np.abs(decision) < threshold):
            return "Unknown"
        else:
            pred_class = svm_model.predict(features)
            class_mapping = {0: "Cat", 1: "Dog"}
            return class_mapping.get(pred_class[0], "Unknown")

    except Exception as e:
        print("Error:", e)
        return None

image_path = r"C:\Users\Ghazal\Desktop\apple.jpg"
result = predict_image(image_path, svm)
print("Predicted class:", result)

In [193]:
X_base_features = get_features(svm_clf,X)
y_base_encoded = y_lable

AttributeError: 'SVC' object has no attribute 'coefs_'

In [27]:
X_horse=[]
Y_horse=[]
if "Horse" in str(folder_name):
    print("horse")
    for horse_img in tqdm(os.listdir(folders_type)):
        horse_image_path = os.path.join(folders_type,horse_img)
        try:
                horse_arr = Open_and_resize(horse_image_path)
                if horse_arr.shape == (128,128,3) and good_image(horse_arr,cls="Horse"):
                    horse_arr_float = horse_arr.flatten().astype('float32')/255.0
                    X_horse.append(horse_arr_float)
                    Y_horse.append("Horse")
        except(UnidentifiedImageError,OSError):
                print("Can't Open",horse_image_path)
X_horse = np.array(X_horse)

horse


100%|█████████████████████████████████████████████████████████████████████████████| 1170/1170 [00:06<00:00, 179.13it/s]


In [28]:
X_horse_features = get_features(MLPModel,X_horse)
y_horse_encoded = np.full(len(X_horse_features),2)

In [8]:
X_all_feauters = np.vstack([X_base_features,X_horse_features])
Y_all_encoded = np.concatenate([y_base_encoded,np.full(len(X_horse_features), 2)])

NameError: name 'X_base_features' is not defined

In [ ]:

clf = LogisticRegression(max_iter=5000)
clf.fit(X_scaled,Y_all_encoded)

In [58]:
def predict_image(image_path,svm):
    try:
        arr_image = Open_and_resize(image_path)
        if arr_image.shape != (128,128,3) and  good_image(arr_image):
            print("Invalid shape:", arr.shape)
            return None
        arr = arr_image.flatten().astype('float32') / 255.0
        arr = arr_image.reshape(1, -1)

        features = get_features(svm, arr)

        pred_class = clf.predict(features)[0]
        print(pred_class)

        # cat=0 , dog=1 , horse=2
        if pred_class == 2:
            return "Horse"
        elif pred_class == 1:
            return "Dog"
        elif pred_class == 0:
            return "Cat"
            
    
    except Exception as e:
        print("Error:", e)
        return None


print(predict_image(r"C:\Users\Ghazal\Desktop\cat.jpg", svm,clf))

Error: 'SVC' object has no attribute 'coefs_'
None


In [ ]:
report = classification_report(y_test,y_predict,target_names=['Cat','Dog'])
print(report)

In [ ]:
# print("X_balanced:", X_balanced.shape)
print("X_horse_features:", X_horse_features.shape)
print("y_horse_encoded:", len(y_horse_encoded))
# print("y:", len(y))

In [35]:
cm = confusion_matrix(y_true, y_predict)
print(cm)

NameError: name 'y_true' is not defined